# MMD Instruction Training — Google Colab
This is an interactive user-started session, not an always-on worker. Upload train.jsonl, validation.jsonl, test.jsonl and provenance.json from a licensed MCT instruction dataset. The model starts from random weights.

In [ ]:
!git clone --depth 1 --branch feat/mmd-cloud-training-mct-02 https://github.com/pctaha501-create/MMD-GameForge.git /content/MMD-GameForge
%pip -q install tokenizers
from google.colab import files
import pathlib, torch
uploaded=files.upload()
DATA_DIR=pathlib.Path('/content/mmd-instruction-data'); DATA_DIR.mkdir(exist_ok=True)
for name,data in uploaded.items(): (DATA_DIR/name).write_bytes(data)
for name in ('train.jsonl','validation.jsonl','test.jsonl','provenance.json'):
    if not (DATA_DIR/name).exists(): raise FileNotFoundError(f'Upload {name}')
REPO=pathlib.Path('/content/MMD-GameForge'); OUTPUT=pathlib.Path('/content/mmd-instruction-run')
print('Device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!python /content/MMD-GameForge/cloud/runtime/model_factory.py create --profile multilingual-tiny --output /content/mmd-instruction-config.json


In [ ]:
!python /content/MMD-GameForge/cloud/runtime/train_instructions.py --config /content/mmd-instruction-config.json --dataset-dir "$DATA_DIR" --output "$OUTPUT"


In [ ]:
!python /content/MMD-GameForge/cloud/runtime/bundle.py --checkpoint-dir "$OUTPUT" --config /content/mmd-instruction-config.json --evaluation "$OUTPUT/instruction-evaluation.json" --output /content/MMD-instruction-model.zip
files.download('/content/MMD-instruction-model.zip')
